In [ ]:
# Malariagen API arguments for each call below.
# Replace a value to override it; keep the marker to use the call-site
# value or the method's default.
API_KEEP_CALLSITE_VALUE = '__KEEP_CALLSITE_VALUE__'
PERFORMANCE_CSV_PATH = 'plot_haplotypes_frequencies_performance.csv'
API_CALL_PARAMETERS = {'cell_3_call_1_ag3_haplotypes_frequencies': {'region': '2L:2,358,158-2,431,617',
                                              'cohorts': 'admin1_year',
                                              'sample_query': None,
                                              'sample_query_options': None,
                                              'min_cohort_size': 10,
                                              'sample_sets': ('AG1000G-BF-A',
                                                              'AG1000G-BF-B',
                                                              'AG1000G-BF-C'),
                                              'chunks': 'native',
                                              'inline_array': True},
 'cell_4_call_1_ag3_plot_frequencies_heatmap': {'df': '__KEEP_CALLSITE_VALUE__',
                                                'index': 'label',
                                                'max_len': 100,
                                                'col_width': 40,
                                                'row_height': 20,
                                                'x_label': 'Cohorts',
                                                'y_label': 'Variants',
                                                'colorbar': True,
                                                'width': None,
                                                'height': None,
                                                'text_auto': '.0%',
                                                'aspect': 'auto',
                                                'color_continuous_scale': 'Reds',
                                                'title': True,
                                                'show': True,
                                                'renderer': None,
                                                'kwargs': '__KEEP_CALLSITE_VALUE__'},
 'cell_5_call_1_ag3_haplotypes_frequencies_advanced': {'region': '2L:2,358,158-2,431,617',
                                                       'area_by': 'admin1_iso',
                                                       'period_by': 'year',
                                                       'sample_sets': ['AG1000G-BF-A',
                                                                       'AG1000G-BF-B'],
                                                       'sample_query': None,
                                                       'sample_query_options': None,
                                                       'min_cohort_size': 10,
                                                       'ci_method': 'wilson',
                                                       'chunks': 'native',
                                                       'inline_array': True,
                                                       'taxon_by': 'taxon',
                                                       'filter_unassigned': None},
 'cell_6_call_1_ag3_plot_frequencies_time_series': {'ds': '__KEEP_CALLSITE_VALUE__',
                                                    'height': None,
                                                    'width': None,
                                                    'title': True,
                                                    'legend_sizing': 'constant',
                                                    'show': True,
                                                    'renderer': None,
                                                    'taxa': None,
                                                    'areas': None,
                                                    'kwargs': '__KEEP_CALLSITE_VALUE__'},
 'cell_9_call_1_af1_haplotypes_frequencies_advanced': {'region': '2RL:2,358,158-2,431,617',
                                                       'area_by': 'admin1_iso',
                                                       'period_by': 'year',
                                                       'sample_sets': ['1232-VO-KE-OCHOMO-VMF00044'],
                                                       'sample_query': None,
                                                       'sample_query_options': None,
                                                       'min_cohort_size': 10,
                                                       'ci_method': 'wilson',
                                                       'chunks': 'native',
                                                       'inline_array': True,
                                                       'taxon_by': 'taxon',
                                                       'filter_unassigned': None},
 'cell_9_call_2_af1_plot_frequencies_time_series': {'ds': '__KEEP_CALLSITE_VALUE__',
                                                    'height': None,
                                                    'width': None,
                                                    'title': True,
                                                    'legend_sizing': 'constant',
                                                    'show': True,
                                                    'renderer': None,
                                                    'taxa': None,
                                                    'areas': None,
                                                    'kwargs': '__KEEP_CALLSITE_VALUE__'}}

In [ ]:
import inspect
from memory_profiler import memory_usage
import time
import json
import csv
from pathlib import Path

def _parameter_value_for_csv(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): _parameter_value_for_csv(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_parameter_value_for_csv(item) for item in value]
    shape = getattr(value, "shape", None)
    if shape is not None:
        return {
            "type": f"{type(value).__module__}.{type(value).__qualname__}",
            "shape": list(shape),
        }
    return repr(value)


import collections.abc
import types
import typing


def _tuple_if_expected(annotation, value):
    """Turn a list back into a tuple where the API accepts a tuple but not a
    list, e.g. percentiles. CSV configs store values as JSON, which has no
    tuples, so (50, 75, 100) arrives as [50, 75, 100]."""
    if not isinstance(value, list):
        return value
    if typing.get_origin(annotation) is typing.Annotated:
        annotation = typing.get_args(annotation)[0]
    if typing.get_origin(annotation) in (typing.Union, types.UnionType):
        options = typing.get_args(annotation)
    else:
        options = (annotation,)
    accepted = {typing.get_origin(option) or option for option in options}
    accepts_list = accepted & {
        list, typing.Any, collections.abc.Sequence, collections.abc.Iterable,
        collections.abc.Collection, collections.abc.MutableSequence,
    }
    if tuple in accepted and not accepts_list:
        return tuple(value)
    return value


def _call_api_with_parameters(method, call_key, *args, **kwargs):
    signature = inspect.signature(method)
    bound = signature.bind_partial(*args, **kwargs)
    bound.apply_defaults()
    for name, value in API_CALL_PARAMETERS[call_key].items():
        if not (isinstance(value, str) and value == API_KEEP_CALLSITE_VALUE):
            parameter = signature.parameters.get(name)
            if parameter is not None:
                value = _tuple_if_expected(parameter.annotation, value)
            bound.arguments[name] = value

    positional = []
    keyword = {}
    for parameter in signature.parameters.values():
        if parameter.kind is inspect.Parameter.POSITIONAL_ONLY:
            if parameter.name in bound.arguments:
                positional.append(bound.arguments[parameter.name])
        elif parameter.kind is inspect.Parameter.POSITIONAL_OR_KEYWORD:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_POSITIONAL:
            positional.extend(bound.arguments.get(parameter.name, ()))
        elif parameter.kind is inspect.Parameter.KEYWORD_ONLY:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_KEYWORD:
            keyword.update(bound.arguments.get(parameter.name, {}))
    result = {}

    def operation():
        result["value"] = method(*positional, **keyword)

    start_time = time.perf_counter()
    peak_memory_mib = memory_usage((operation, ()), max_usage=True)
    runtime_sec = time.perf_counter() - start_time

    output_path = Path(PERFORMANCE_CSV_PATH)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "call_key",
        "method",
        "parameters_json",
        "runtime_sec",
        "peak_memory_mib",
    ]
    record = {
        "call_key": call_key,
        "method": getattr(method, "__qualname__", getattr(method, "__name__", repr(method))),
        "parameters_json": json.dumps(
            {
                name: _parameter_value_for_csv(value)
                for name, value in bound.arguments.items()
            },
            sort_keys=True,
        ),
        "runtime_sec": round(runtime_sec, 4),
        "peak_memory_mib": round(peak_memory_mib, 2),
    }
    file_exists = output_path.exists() and output_path.stat().st_size > 0
    with output_path.open("a", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(record)

    return result["value"]

# Ag3

In [ ]:
import malariagen_data

ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../gcs_cache"),
    cohorts_analysis="20230516",
)
ag3

In [ ]:
hap_df = _call_api_with_parameters(ag3.haplotypes_frequencies, 'cell_3_call_1_ag3_haplotypes_frequencies', "2L:2,358,158-2,431,617", "admin1_year", sample_sets=("AG1000G-BF-A", "AG1000G-BF-B", "AG1000G-BF-C"))
hap_df

In [ ]:
_call_api_with_parameters(ag3.plot_frequencies_heatmap, 'cell_4_call_1_ag3_plot_frequencies_heatmap', hap_df.query("max_af > .05"))

In [ ]:
hap_xr = _call_api_with_parameters(ag3.haplotypes_frequencies_advanced, 'cell_5_call_1_ag3_haplotypes_frequencies_advanced', region="2L:2,358,158-2,431,617", area_by="admin1_iso", period_by="year", sample_sets=["AG1000G-BF-A", "AG1000G-BF-B"])

In [ ]:
_call_api_with_parameters(ag3.plot_frequencies_time_series, 'cell_6_call_1_ag3_plot_frequencies_time_series', hap_xr)

# Af1

In [ ]:
af1 = malariagen_data.Af1(
    "simplecache::gs://vo_afun_release_master_us_central1",
    simplecache=dict(cache_storage="../gcs_cache"),
)
af1

In [ ]:
hap_xr = _call_api_with_parameters(af1.haplotypes_frequencies_advanced, 'cell_9_call_1_af1_haplotypes_frequencies_advanced', region="2RL:2,358,158-2,431,617", area_by="admin1_iso", period_by="year", sample_sets=["1232-VO-KE-OCHOMO-VMF00044"], min_cohort_size=10)
_call_api_with_parameters(af1.plot_frequencies_time_series, 'cell_9_call_2_af1_plot_frequencies_time_series', hap_xr)